In [2]:
!git clone https://github.com/mateenyaqoob/unstructuredata_tfidf_repo.git
%cd unstructuredata_tfidf_repo
!pip -q install -r requirements.txt
!pip -q install nltk

Cloning into 'unstructuredata_tfidf_repo'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 54 (delta 18), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 25.71 KiB | 4.29 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/unstructuredata_tfidf_repo
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 97.6 MB/s eta 0:00:00


In [3]:
import nltk
for pkg in ["stopwords", "punkt", "punkt_tab"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Could not download {pkg}: {e}")

In [4]:
import os
import sys
import importlib.util
from nltk.tokenize import word_tokenize

REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Path to the dataset file in the repo
DATA_MODULE_PATH = os.path.join(REPO_ROOT, "data", "documents.py")

# Load data/documents.py as a module without hardcoding its contents
spec = importlib.util.spec_from_file_location("documents", DATA_MODULE_PATH)
documents_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(documents_module)

TITLES = documents_module.TITLES
DOCUMENTS = documents_module.DOCUMENTS

print(f"{len(DOCUMENTS)} documents loaded.\n")
for title, doc in zip(TITLES, DOCUMENTS):
    nltk_tokens = word_tokenize(doc)
    print(f"  - {title}: {len(doc.split())} words / {len(nltk_tokens)} NLTK tokens")

10 documents loaded.

  - AI in Healthcare: 85 words / 101 NLTK tokens
  - Climate Change: 83 words / 97 NLTK tokens
  - Basketball Strategy: 85 words / 102 NLTK tokens
  - Italian Pasta: 91 words / 108 NLTK tokens
  - Space Exploration: 86 words / 101 NLTK tokens
  - Cybersecurity: 88 words / 109 NLTK tokens
  - Music Theory: 87 words / 111 NLTK tokens
  - Gardening Vegetables: 89 words / 108 NLTK tokens
  - Economics Inflation: 89 words / 109 NLTK tokens
  - Public Health: 91 words / 115 NLTK tokens


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(DOCUMENTS)

print("Matrix shape:", X.shape)  # expected: (10, n_unique_terms)
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))

Matrix shape: (10, 537)
Vocabulary size: 537


In [6]:
import pandas as pd

df = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=TITLES,
)

print("Sample of TF-IDF matrix (first 3 docs × 10 terms):")
print(df.iloc[:3, :10].round(3))

print("\nTop 5 TF-IDF terms per document:")
for title in TITLES:
    top5 = df.loc[title].sort_values(ascending=False).head(5)
    print(f"\n{title}:")
    for term, score in top5.items():
        print(f"  {term:<20s} {score:.3f}")


Sample of TF-IDF matrix (first 3 docs × 10 terms):
                     ability  accelerate  access  accountability  accuracy  \
AI in Healthcare        0.00         0.0     0.0           0.125     0.125   
Climate Change          0.00         0.0     0.0           0.000     0.000   
Basketball Strategy     0.13         0.0     0.0           0.000     0.000   

                     accurate  actions  activity  adaptation  add  
AI in Healthcare        0.000    0.000       0.0       0.000  0.0  
Climate Change          0.125    0.125       0.0       0.125  0.0  
Basketball Strategy     0.000    0.000       0.0       0.000  0.0  

Top 5 TF-IDF terms per document:

AI in Healthcare:
  ai                   0.374
  healthcare           0.249
  use                  0.165
  waiting              0.125
  transforming         0.125

Climate Change:
  change               0.250
  emissions            0.250
  energy               0.212
  climate              0.212
  burning              0.125

Bas

In [7]:
# ============================================================
# Cell 5 — Step 4: Query similarity challenge


from sklearn.metrics.pairwise import cosine_similarity

def rank_documents(query: str) -> pd.DataFrame:
    """Return documents ranked by cosine similarity to the query."""
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, X).flatten()
    ranking = (
        pd.DataFrame({"title": TITLES, "similarity": scores})
        .sort_values("similarity", ascending=False)
        .reset_index(drop=True)
    )
    print(f"\n=== Query: {query!r} ===")
    print(ranking.to_string(index=False))
    return ranking

queries = [
    "renewable energy and climate policy",
    "password security and phishing",
    "healthy sleep exercise and diet",
]

for q in queries:
    rank_documents(q)


=== Query: 'renewable energy and climate policy' ===
               title  similarity
      Climate Change    0.323458
 Economics Inflation    0.095225
Gardening Vegetables    0.048458
    AI in Healthcare    0.000000
 Basketball Strategy    0.000000
       Italian Pasta    0.000000
       Cybersecurity    0.000000
   Space Exploration    0.000000
        Music Theory    0.000000
       Public Health    0.000000

=== Query: 'password security and phishing' ===
               title  similarity
       Cybersecurity    0.176375
    AI in Healthcare    0.000000
      Climate Change    0.000000
 Basketball Strategy    0.000000
       Italian Pasta    0.000000
   Space Exploration    0.000000
        Music Theory    0.000000
Gardening Vegetables    0.000000
 Economics Inflation    0.000000
       Public Health    0.000000

=== Query: 'healthy sleep exercise and diet' ===
               title  similarity
       Public Health    0.284861
Gardening Vegetables    0.044702
      Climate Change  

In [8]:
# ============================================================
# Cell 6 — Step 5: Optional challenge with n-grams and thresholds
# ============================================================

vectorizer_v2 = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2,
)
X_v2 = vectorizer_v2.fit_transform(DOCUMENTS)

print("v2 Matrix shape:", X_v2.shape)
print("v2 Vocabulary size:", len(vectorizer_v2.get_feature_names_out()))
print("v1 Vocabulary size:", len(vectorizer.get_feature_names_out()))

df_v2 = pd.DataFrame(
    X_v2.toarray(),
    columns=vectorizer_v2.get_feature_names_out(),
    index=TITLES,
)

print("\nTop 5 v2 terms for first 3 documents:")
for title in TITLES[:3]:
    top5 = df_v2.loc[title].sort_values(ascending=False).head(5)
    print(f"\n{title}:")
    for term, score in top5.items():
        print(f"  {term:<25s} {score:.3f}")

v2 Matrix shape: (10, 69)
v2 Vocabulary size: 69
v1 Vocabulary size: 537

Top 5 v2 terms for first 3 documents:

AI in Healthcare:
  use                       0.391
  clean                     0.252
  intelligence              0.252
  disease                   0.252
  careful                   0.252

Climate Change:
  climate                   0.386
  energy                    0.386
  governments               0.193
  efficiency                0.193
  global                    0.193

Basketball Strategy:
  adjust                    0.300
  based                     0.300
  defenses                  0.300
  decisions                 0.300
  communication             0.300


In [9]:
# Cell 7 — Step 6: Verify with tests + manual assertions


# Run provided unit tests
!pytest -q

# Manual verification
assert len(DOCUMENTS) == 10, "Expected 10 documents"
assert all(len(doc.split()) > 30 for doc in DOCUMENTS), "Each document should be long text (>30 words)"
assert X.shape[0] == 10, "TF-IDF matrix should have 10 rows"
assert X.shape[1] == len(vectorizer.get_feature_names_out()), "Column count should equal vocabulary size"

top = rank_documents("renewable energy and climate policy")
assert top.iloc[0]["title"] == "Climate Change", "Climate Change should rank first"

print("\nAll manual checks passed.")

...                                                                      [100%]
3 passed in 1.29s

=== Query: 'renewable energy and climate policy' ===
               title  similarity
      Climate Change    0.323458
 Economics Inflation    0.095225
Gardening Vegetables    0.048458
    AI in Healthcare    0.000000
 Basketball Strategy    0.000000
       Italian Pasta    0.000000
       Cybersecurity    0.000000
   Space Exploration    0.000000
        Music Theory    0.000000
       Public Health    0.000000

All manual checks passed.
